In [2]:
import ee
import geopandas as gpd
import json

# -------------------------
# Reclass maps (keep yours)
# -------------------------
RECLASS_MAP_C3S = {
    50: 1, 60: 1, 61: 1, 62: 1, 70: 1, 71: 1, 72: 1, 80: 1, 81: 1, 82: 1, 90: 1, 100: 1,
    110: 2, 120: 2, 121: 2, 122: 2, 130: 2, 140: 2, 150: 2, 151: 2, 152: 2, 153: 2,
    10: 3, 11: 3, 12: 3, 20: 3, 30: 3, 40: 3,
    160: 4, 170: 4, 180: 4,
    190: 5,
    200: 6, 201: 6, 202: 6, 220: 6,
    210: 7,
    0: 0,
}
C3S_FROM = list(RECLASS_MAP_C3S.keys())
C3S_TO   = list(RECLASS_MAP_C3S.values())

RECLASS_MAP_ESRI = {
    2: 1,    # Trees -> Forest
    11: 2,   # Rangeland -> Grassland
    5: 3,    # Crops -> Cropland
    4: 4,    # Flooded vegetation -> Wetlands
    7: 5,    # Built area -> Artificial
    8: 6,    # Bare ground -> Other
    1: 7,    # Water -> Water
    10: 0,   # Clouds -> No data
    9: 6,    # Snow/Ice -> Other
}
ESRI_FROM = list(RECLASS_MAP_ESRI.keys())
ESRI_TO   = list(RECLASS_MAP_ESRI.values())


def load_aoi_to_ee(aoi_shp: str):
    """
    Read AOI shapefile -> ee.FeatureCollection + ee.Geometry (union)
    Assumes shp has polygons; will reproject to EPSG:4326.
    """
    gdf = gpd.read_file(aoi_shp)
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)
    fc = ee.FeatureCollection(json.loads(gdf.to_json()))
    return fc, fc.geometry()


def get_collections_and_meta(geom: ee.Geometry):
    """
    Return collections and detected band names, plus target projection from C3S.
    """
    LC_C3S  = "projects/sat-io/open-datasets/ESA/C3S-LC-L4-LCCS"
    LC_ESRI = "projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS"

    c3s_ic_all  = ee.ImageCollection(LC_C3S).filterBounds(geom)
    esri_ic_all = ee.ImageCollection(LC_ESRI).filterBounds(geom)

    c3s_bands = c3s_ic_all.first().bandNames().getInfo()
    c3s_band = "lccs_class" if "lccs_class" in c3s_bands else c3s_bands[0]

    esri_bands = esri_ic_all.first().bandNames().getInfo()
    esri_band = esri_bands[0]

    target_proj = c3s_ic_all.first().select(c3s_band).projection()  # 300m grid reference
    return c3s_ic_all, esri_ic_all, c3s_band, esri_band, target_proj


def get_reclass_image(
    *,
    year: int,
    source: str,
    geom: ee.Geometry,
    scale: int = 300,
    c3s_ic_all: ee.ImageCollection = None,
    esri_ic_all: ee.ImageCollection = None,
    c3s_band: str = None,
    esri_band: str = None,
    target_proj: ee.Projection = None,
    mask_zero: bool = True,
) -> ee.Image:
    """
    Return an ee.Image with band 'cls' (int16, values 0..7) that is:
      - clipped to geom
      - reclassified to your scheme
      - aligned to target_proj at `scale` (default 300m)
    source: "C3S" or "ESRI"
    """

    source = source.upper().strip()
    if source not in {"C3S", "ESRI"}:
        raise ValueError("source must be 'C3S' or 'ESRI'")

    # Allow passing preloaded collections/meta for speed; otherwise detect on the fly
    if any(v is None for v in [c3s_ic_all, esri_ic_all, c3s_band, esri_band, target_proj]):
        c3s_ic_all, esri_ic_all, c3s_band, esri_band, target_proj = get_collections_and_meta(geom)

    if source == "C3S":
        img = (c3s_ic_all
               .filterDate(f"{year}-01-01", f"{year}-12-31")
               .first())
        cls = (ee.Image(img)
               .select(c3s_band)
               .clip(geom)
               .remap(C3S_FROM, C3S_TO)
               .rename("cls")
               .toInt16()
               .reproject(crs=target_proj.crs(), scale=scale))

    else:  # ESRI
        ic_y = esri_ic_all.filterDate(f"{year}-01-01", f"{year}-12-31")
        # avoid client-side getInfo if you want fully server-side; but missing-year check is useful
        if ic_y.size().getInfo() == 0:
            raise ValueError(f"ESRI year {year} has no data in this AOI.")

        img_y = ic_y.mosaic()

        # Use true 10m projection from first tile, then downscale to 300m
        proj10 = ee.Image(ic_y.first()).select(esri_band).projection()
        img_y = img_y.setDefaultProjection(proj10)

        cls10 = (img_y
                .select(esri_band)
                .clip(geom)
                .toInt16()
                .setDefaultProjection(proj10))

        cls300 = (cls10
                  .reduceResolution(reducer=ee.Reducer.mode(), maxPixels=4096)
                  .reproject(crs=target_proj.crs(), scale=scale))

        cls = (cls300
               .remap(ESRI_FROM, ESRI_TO)
               .rename("cls")
               .toInt16()
               .reproject(crs=target_proj.crs(), scale=scale))

    if mask_zero:
        cls = cls.updateMask(cls.neq(0))

    return cls


In [4]:
import geemap
import ee

PROJECT_ID = "celestial-air-485003-n1"
AOI_SHP = r"../../data_hmq/Map/abu_dhabi_all.shp"

ee.Initialize(project=PROJECT_ID)

aoi_fc, geom = load_aoi_to_ee(AOI_SHP)

# （可选）预加载集合/元信息：多次调用会快很多
c3s_ic_all, esri_ic_all, c3s_band, esri_band, target_proj = get_collections_and_meta(geom)

# 取 2000 年 C3S 重分类影像
cls_img = get_reclass_image(
    year=2000,
    source="C3S",
    geom=geom,
    scale=300,
    c3s_ic_all=c3s_ic_all,
    esri_ic_all=esri_ic_all,
    c3s_band=c3s_band,
    esri_band=esri_band,
    target_proj=target_proj,
    mask_zero=True
)

palette = [
    "000000",  # 0 (masked)
    "2e7d32",  # 1 Tree-covered
    "fdae61",  # 2 Grassland
    "ffff66",  # 3 Cropland
    "00c853",  # 4 Wetland
    "e41a1c",  # 5 Artificial
    "b8860b",  # 6 Other
    "0b4fa2",  # 7 Water
]

m = geemap.Map()
m.centerObject(aoi_fc, 7)

m.addLayer(cls_img, {"min": 1, "max": 7, "palette": palette[1:]}, "C3S 2000 (reclass)")
m.addLayer(aoi_fc.style(**{"color": "000000", "fillColor": "00000000", "width": 2}), {}, "AOI")
m


EEException: Caller does not have required permission to use project celestial-air-485003-n1. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam/project?project=celestial-air-485003-n1 and then retry. Propagation of the new permission may take a few minutes.